# 01 — Data Auditing

# Overview

This notebook performs the initial data audit for the **Higher Education Outcomes Analysis** project, using **2024 C1** as the first analytical cohort.

The purpose of this stage is to establish a reliable understanding of the structure, quality, and integration readiness of the project's core datasets before applying transformations, feature engineering, or analytical modeling.

The audit covers three structurally linked datasets:

- **Enrollment (primary fact table):** academic outcome records at the `(course_code, section)` level, including enrollment volume and final student status distributions (dropout, insufficient performance, free status, regular completion, and promoted completion).
- **Programs (reference table):** canonical mapping between `program_code` and program descriptors, used to enrich the main analytical table with program-level context.
- **Offering (operational metadata table):** course-section operational attributes for the selected period, including weekday, shift, delivery mode, and campus assignment.

This notebook focuses on:

- schema inspection and type validation,
- duplicate detection based on natural keys,
- missing value profiling,
- categorical cardinality and semantic consistency checks,
- categorical distribution profiling,
- dataset-level key uniqueness validation,
- cross-dataset referential integrity checks,
- merge coverage assessment across source tables.

A specific audit objective is to evaluate the completeness and reliability of operational metadata originally present in the Enrollment dataset and validate the external Offering dataset as its canonical replacement source for downstream analysis.

The public version of this project uses a **deterministically pseudonymized but structurally equivalent dataset**, preserving relational integrity, statistical properties, and analytical fidelity while protecting institutional confidentiality.

In [1]:
from notebook_utils import ensure_repo_root

# Establish the repository root as the working directory for this notebook
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_data, set_pandas_display_options

enrollment = load_data("enrollment")
programs = load_data("programs")
offering = load_data("offering")

# Set pandas display options for better readability
set_pandas_display_options()


In [3]:
from src.config.contracts import (
    ENROLLMENT_SCHEMA,
    OFFERING_SCHEMA,
    PROGRAMS_SCHEMA,
    PRIMARY_KEYS,
)
from src.auditing import (
    inspect_schema,
    check_key_uniqueness,
    check_duplicates,
    check_one_to_one_mapping,
    profile_missingness,
    profile_categoricals,
    check_referential_integrity,
    validate_metric_consistency,
    )

## Dataset schema inspection

This section validates the structural integrity of the three analytical sources by comparing observed dataframe schemas against predefined data contracts.

The inspection focuses on:
- column presence and naming consistency,
- dtype alignment,
- missing value distribution,
- and categorical cardinality.

These checks establish whether the datasets are structurally suitable for downstream cleaning, integration, and analysis workflows.


### Enrollment

In [4]:
inspect_schema(enrollment, ENROLLMENT_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,campus,categorical,object,object,True,117,61.39,12
1,schedule_time,categorical,object,object,True,122,59.74,22
2,delivery_mode,categorical,object,object,True,122,59.74,11
3,weekday,categorical,object,object,True,122,59.74,6
4,shift,categorical,object,object,True,122,59.74,5
5,course_name,descriptor,object,object,True,303,0.00,131
6,course_code,identifier,object,object,True,303,0.00,131
7,total_enrollment,metric,int64,int64,True,303,0.00,88
8,free_status_count,metric,int64,int64,True,303,0.00,53
9,regular_completion_count,metric,int64,int64,True,303,0.00,53


### Offering

In [5]:
inspect_schema(offering, OFFERING_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,course_code,identifier,object,object,True,305,0.00,138
1,schedule_time,categorical,object,object,True,305,0.00,38
2,delivery_mode,categorical,object,object,True,305,0.00,26
3,section,identifier,int64,int64,True,305,0.00,16
4,weekday,categorical,object,object,True,305,0.00,14
5,shift,categorical,object,object,True,305,0.00,6
6,campus,categorical,object,object,True,305,0.00,6
7,workload,metric,float64,int64,False,305,0.00,4


### Programs

In [6]:
inspect_schema(programs, PROGRAMS_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,program_code,foreign_key,object,object,True,13,0.00,13
1,program_name,descriptor,object,object,True,13,0.00,13


### Findings

- All datasets conform broadly to their expected analytical schemas.
- Identifier and metric fields exhibit consistent typing across sources.
- Enrollment presents substantial missingness in operational metadata fields (`shift`, `weekday`, `delivery_mode`, `campus`), limiting its reliability as a standalone source for scheduling information.
- Offering categorical fields exhibit manageable but non-trivial cardinality, with several categories requiring normalization due to formatting and semantic fragmentation.
- `workload` was loaded as `float` despite representing an integer-valued metric, requiring dtype coercion during preprocessing.
- No major structural anomalies were detected at the schema level.

## Primary Keys Uniqueness Validation

This section validates the uniqueness of the analytical primary keys defined for each dataset.

The checks ensure that:
- each observation represents a unique real-world entity,
- no duplicated identifiers exist,
- and downstream joins can be performed without unintended row multiplication.

For Enrollment and Offering, uniqueness is validated over the composite key (`course_code`, `section`), while Programs is validated over `program_code`.

### Enrollment

In [7]:
check_key_uniqueness(enrollment, PRIMARY_KEYS["enrollment"])

Key is unique: course_code, section


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,"course_code, section",303,303,0,True


### Offering

In [8]:
check_key_uniqueness(offering, PRIMARY_KEYS["offering"])

Key is unique: course_code, section


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,"course_code, section",305,305,0,True


### Programs

In [9]:
check_key_uniqueness(programs, PRIMARY_KEYS["programs"])

Key is unique: program_code


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,program_code,13,13,0,True


### Findings

- No primary key violations were detected across the three datasets.
- Enrollment and Offering preserve a consistent one-row-per-course-section structure.
- Programs maintains a stable one-to-one mapping between program identifiers and program labels.
- The absence of duplicated keys supports reliable dataset integration and downstream relational operations.

## Full-row duplicate assessment

This section evaluates whether identical records are repeated within the Enrollment and Offering datasets.

The objective is to detect potential duplication introduced during extraction, export, or consolidation processes.

### Enrollment

In [10]:
check_duplicates(enrollment, enrollment.columns.to_list())

No duplicates found for subset: course_name, section, total_enrollment, dropout_count, insufficient_count, free_status_count, promoted_completion_count, regular_completion_count, course_code, program_code, shift, weekday, schedule_time, delivery_mode, campus


""


### Offering

In [11]:
check_duplicates(offering, offering.columns.to_list())

No duplicates found for subset: course_code, section, workload, shift, weekday, schedule_time, delivery_mode, campus


""


### Findings

- No fully duplicated rows were detected in either dataset.
- The absence of exact record replication supports the integrity of the raw observational structure.

## Code-name consistency validation

This section validates the semantic consistency between identifier fields and their associated labels.

The objective is to ensure that each `course_code` and `program_code` maps consistently to a single descriptive name across observations.

### Course Code - Course Name

In [12]:
check_one_to_one_mapping(enrollment, "course_code", "course_name",)

No inconsistencies detected between `course_code` and `course_name`.


""


### Course Name - Course Code

In [13]:
check_one_to_one_mapping(enrollment, "course_name", "course_code")

No inconsistencies detected between `course_name` and `course_code`.


""


### Program Code - Program Name

In [14]:
check_one_to_one_mapping(programs, "program_code", "program_name")

No inconsistencies detected between `program_code` and `program_name`.


""


### Program Name - Program Code

In [15]:
check_one_to_one_mapping(programs, "program_name", "program_code")

No inconsistencies detected between `program_name` and `program_code`.


""


### Findings

- No inconsistencies were detected between course or program identifiers and their associated labels.
- Both mappings exhibit stable one-to-one relationships, supporting reliable categorical interpretation and downstream joins.

## Missing value profiling

This section evaluates missing value distribution across the analytical datasets, with particular attention to operational metadata completeness in Enrollment.

The objective is to assess field-level reliability, identify structurally sparse variables, and validate the need for external metadata enrichment through the Offering dataset.

In [16]:
profile_missingness(enrollment)

5 critical and 0 warning-level columns detected.


,column,null_count,null_pct,status
0,campus,186,61.39,critical
1,shift,181,59.74,critical
2,weekday,181,59.74,critical
3,schedule_time,181,59.74,critical
4,delivery_mode,181,59.74,critical
5,section,0,0.00,ok
6,course_name,0,0.00,ok
7,promoted_completion_count,0,0.00,ok
8,free_status_count,0,0.00,ok
9,insufficient_count,0,0.00,ok


In [17]:
profile_missingness(offering)

0 critical and 0 warning-level columns detected.


,column,null_count,null_pct,status
0,course_code,0,0.00,ok
1,section,0,0.00,ok
2,workload,0,0.00,ok
3,shift,0,0.00,ok
4,weekday,0,0.00,ok
5,schedule_time,0,0.00,ok
6,delivery_mode,0,0.00,ok
7,campus,0,0.00,ok


### Findings

- Enrollment exhibits substantial missingness in native operational metadata fields, particularly `shift`, `weekday`, `delivery_mode`, and `campus`.
- The observed sparsity pattern suggests that scheduling metadata was not consistently maintained within the original Enrollment source.
- In contrast, Offering presents complete coverage across operational metadata attributes, supporting its role as the canonical metadata source for downstream integration.
- Metric and identifier fields remain fully populated across the analytical base tables.

## Categorical distribution profiling

This section examines the distribution and cardinality of key categorical variables within the operational metadata fields.

The objective is to identify potential semantic fragmentation, inconsistent labeling patterns, and category structures requiring normalization before analytical integration.

In [18]:
categorical_profiles = profile_categoricals(
    offering,
    columns=[
        "shift",
        "weekday",
        "delivery_mode",
        "campus",
    ]
)

`shift` → 6 unique categories detected.
`weekday` → 14 unique categories detected.
`delivery_mode` → 26 unique categories detected.
`campus` → 6 unique categories detected.


In [19]:
for col, profile in categorical_profiles.items():
    print(f"Column: {col}")
    display(profile["top_categories"])

Column: shift


,shift,count,pct
0,noche,181,59.34
1,mañana,68,22.30
2,tarde,50,16.39
3,mañana,4,1.31
4,mañama,1,0.33
5,Noche,1,0.33


Column: weekday


,weekday,count,pct
0,martes,63,20.66
1,jueves,63,20.66
2,miercoles,52,17.05
3,lunes,48,15.74
4,viernes,45,14.75
5,sabado,18,5.90
6,Miercoles,4,1.31
7,miercoles,4,1.31
8,viernes,2,0.66
9,Jueves,2,0.66


Column: delivery_mode


,delivery_mode,count,pct
0,presencial,105,34.43
1,presencial,69,22.62
2,virtual,29,9.51
3,4 hs presencial y 2 virtual,22,7.21
4,Presencial,18,5.90
5,virtual,15,4.92
6,peesencial,10,3.28
7,virtual (con encuentros presenciales),5,1.64
8,4 hs presenciales y 2 virtuales,4,1.31
9,presencial y 2 hs virtual,4,1.31


Column: campus


,campus,count,pct
0,CAMPUS_001,159,52.13
1,CAMPUS_004,58,19.02
2,CAMPUS_006,44,14.43
3,CAMPUS_003,23,7.54
4,CAMPUS_002,18,5.90
5,CAMPUS_005,3,0.98


### Findings

- Operational metadata fields exhibit controlled and interpretable category distributions across the analytical base.
- `shift` and `campus` maintain low-cardinality structures with limited semantic variation.
- Higher-cardinality fields such as `schedule_time` and `delivery_mode` may require targeted normalization during the cleaning stage.
- No evidence of severe categorical fragmentation or structurally anomalous category distributions was detected.

## Referential integrity and merge coverage

This section evaluates key-level compatibility between Enrollment and Offering using the composite identifier (`course_code`, `section`), as well as Enrollment and Programs, under the use of the identifier (`program_code`).

The objective is to assess merge coverage, identify unmatched records across sources, and validate the operational feasibility of consolidating external metadata into the analytical base table.

### Enrollment - Offering

In [20]:
integrity_results  = check_referential_integrity(
    enrollment,
    offering,
    keys=["course_code", "section"],
)

297 matched keys | 6 unmatched left keys | 8 unmatched right keys

Left-only sample (first 10):
course_code  section
    CRS_093        9
    CRS_101        2
    CRS_046        8
    CRS_056       12
    CRS_085       11
    CRS_094        3

Right-only sample (first 10):
course_code  section
    CRS_136        1
    CRS_132        1
    CRS_137        1
    CRS_138        1
    CRS_133        1
    CRS_135        1
    CRS_108        9
    CRS_134        1


### Enrollment - Programs

In [21]:
integrity_enr_prog = check_referential_integrity(
    enrollment,
    programs,
    keys=["program_code"],
)

13 matched keys | 0 unmatched left keys | 0 unmatched right keys

Left-only sample (first 10):
<none>

Right-only sample (first 10):
<none>


### Findings

- The Offering dataset achieves near-complete coverage over Enrollment course-section combinations.
- The high merge compatibility supports the use of Offering as the canonical operational metadata source for downstream enrichment.
- Residual unmatched observations likely reflect inconsistencies in source registration or incomplete operational records for specific course sections.
- On the other hand, the merge between Programs and Enrollment is fully consistent.

## Metric consistency validation

This section validates the internal consistency of enrollment outcome metrics by comparing the reported `total_enrollment` count against the aggregated sum of all final student status categories.

The objective is to detect potential counting inconsistencies, incomplete status allocation, or aggregation errors affecting analytical reliability.

In [22]:
validate_metric_consistency(
    enrollment.iloc[:, :8],
    component_cols=[
        "dropout_count",
        "insufficient_count",
        "free_status_count",
        "promoted_completion_count",
        "regular_completion_count",
    ],
    total_col="total_enrollment",
)

4 inconsistent rows detected against `total_enrollment`.


,course_name,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count,computed_total,difference
0,COURSE 114,2,19,11,0,0,0,7,18,-1
1,COURSE 056,7,51,12,21,11,2,3,49,-2
2,COURSE 130,3,58,3,0,13,0,41,57,-1
3,COURSE 092,1,57,0,5,33,0,18,56,-1


### Findings

- All observations satisfy additive consistency between outcome categories and reported enrollment totals, except for four records showing minor discrepancies (-1, -2), which are likely attributable to typographical errors during data entry.
- The outcome metrics exhibit strong internal coherence, supporting their suitability for downstream analytical modeling and aggregation workflows.

## Lightweight domain validation

This section performs a small set of business-rule checks over the enrollment outcome metrics.

The objective is to identify structurally invalid observations, including negative counts, zero-enrollment course sections and range checks, before downstream analytical processing.

In [23]:
negative_counts = enrollment[
    [
        "total_enrollment",
        "dropout_count",
        "insufficient_count",
        "free_status_count",
        "promoted_completion_count",
        "regular_completion_count",
    ]
].lt(0).any(axis=1)

zero_total_enrollment = enrollment["total_enrollment"] == 0

print(f"Rows with negative counts: {negative_counts.sum()}")
print(f"Rows with zero total enrollment: {zero_total_enrollment.sum()}")

Rows with negative counts: 0
Rows with zero total enrollment: 0


In [24]:
round(enrollment.describe(percentiles=[]), 2)

,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count
count,303.00,303.00,303.00,303.00,303.00,303.00,303.00
mean,3.22,43.66,3.82,3.26,13.71,7.37,15.48
std,3.17,26.08,9.20,8.26,15.06,13.95,14.40
min,1.00,1.00,0.00,0.00,0.00,0.00,0.00
50%,2.00,46.00,0.00,0.00,10.00,0.00,13.00
max,16.00,112.00,55.00,62.00,69.00,81.00,94.00


### Findings

- No negative metric values were detected across enrollment outcome fields.
- No zero-enrollment course sections were identified in the analytical base.
- The observed metric distributions remain consistent with expected academic reporting constraints.

## Next Steps: Data Cleaning and Integration

Based on the audit findings, the raw datasets exhibit structural soundness and reliable primary keys. However, they require specific normalization and integration steps before being suitable for analytical modeling. 

The subsequent data cleaning pipeline (`02_data_cleaning.ipynb`) will implement the following tasks:

### 1. Integration and Feature Enrichment
- **Metadata Consolidation:** Drop the structurally sparse operational columns (`campus`, `shift`, `weekday`, `schedule_time`, `delivery_mode`) from **Enrollment**.
- **Join Operations:** Perform a left join using `(course_code, section)` to append the canonical operational metadata from **Offering** into **Enrollment**. Merge **Programs** via `program_code` to include program descriptors.
- **Handling Unmatched Records:** Define a resolution strategy (e.g., imputation or exclusion) for the 6 `Enrollment` observations that failed to match with `Offering`, as they will otherwise introduce new missing values into the enriched dataset.

### 2. Categorical Normalization
- **Text Standardization:** Correct typographical errors and unify text casing across categorical features in the Offering dataset (e.g., resolving "peesencial" to "presencial", standardizing "mañama" and "Noche").
- **Taxonomy Consolidation:** Group the fragmented and high-cardinality values found in `delivery_mode`, `shift`, `weekday` and `schedule_time`  into a cleaner, simplified taxonomy.

### 3. Data Type Coercion and Consistency
- **Type Casting:** Coerce the `workload` variable from `float64` to `int64` to match its expected metric contract.
- **Metric Reconciliation:** Address the minor additive discrepancies detected in the outcome metrics (off-by-1 or 2 errors) by either recomputing `total_enrollment` from its components or documenting the margin of error.
